# 01 Graphs and Features

**Cold-Start Social-Graph Leakage: Simulation Study**

This notebook generates realistic social network data and **saves it to Google Drive.** The next notebook (02 Recommender Training) reads the data directly from Drive. This way, the notebooks can be uploaded separately and run in sequence.

### Design principle: breaking circularity
We are **not designing** the leakage. This layer only produces *realistic structure*: a directed "follow" graph (`u → v` = u follows v), `public/private` labels, and node features with **adjustable** correlation to community structure (legitimate interest/geography proxies). The leakage will later emerge on its own from a *standard* recommendation system, *not designed to leak*.

### The real payoff of the simulation: **ground-truth**
The target's true neighborhood `N(t) = followers ∪ followees` is fully known here → precision **and** recall can be reported later. No live API, real private account, or ToS violation involved.

## 0. Setup and Mounting Google Drive

In [ ]:
!pip install -q networkx numpy pandas scipy scikit-learn matplotlib

In [ ]:
# --- repository-root anchor (added when notebooks moved into notebooks/) ---
# Notebooks live in notebooks/ but read and write cold_start_leakage/ at the
# repository root, so resolve that root explicitly instead of relying on cwd.
import os as _os_anchor
_here = _os_anchor.getcwd()
if _os_anchor.path.basename(_here) == 'notebooks':
    _os_anchor.chdir(_os_anchor.path.dirname(_here))
# --- end anchor ---
import os, json, gzip, pickle, urllib.request
from dataclasses import dataclass, field, asdict
import numpy as np
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

# --- Mount Google Drive (Colab) ---
def _in_colab():
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False

if _in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/cold_start_leakage/data'
else:
    DATA_DIR = os.path.abspath('./cold_start_leakage/data')   # Non-Colab (local test)

os.makedirs(DATA_DIR, exist_ok=True)
print("Data will be saved here:", DATA_DIR)
print("networkx", nx.__version__)

## 1. Configuration

All "buttons" in one place. `homophily` = legitimate signal strength (0 = random feature, 1 = fully disjoint); the basis for the ablation later on.

In [ ]:
@dataclass
class DataConfig:
    name: str = "sbm_default"
    seed: int = 42
    private_ratio: float = 0.30           # private (closed) account ratio
    private_mode: str = "random"          # "random" | "low_degree"
    mutual_ratio: float = 0.49            # mutual follow probability (~Table VI)
    feature_dim: int = 16
    homophily: float = 0.70               # legitimate signal strength
    # SBM
    sbm_sizes: tuple = (180, 160, 140, 120, 100)
    sbm_p_in: float = 0.06
    sbm_p_out: float = 0.0015
    # LFR
    lfr_n: int = 700
    lfr_tau1: float = 2.7
    lfr_tau2: float = 1.4
    lfr_mu: float = 0.18
    lfr_avg_degree: int = 12
    lfr_min_community: int = 60
    # real graphs (SNAP), Reviewer 3 item 1: "a few more real datasets"
    snap_url: str = "https://snap.stanford.edu/data/facebook_combined.txt.gz"
    twitter_url: str = "https://snap.stanford.edu/data/twitter_combined.txt.gz"
    email_url: str = "https://snap.stanford.edu/data/email-Eu-core.txt.gz"
    real_node_cap: int = 5000             # Downsample large real graphs like Twitter to this size using BFS

## 2. Standard dataset container (schema)

The next notebook relies on these fields: `G` (directed graph), `private`, `community`, `node_list`, `features`. Ground-truth helpers: `followers / following / neighborhood`.

In [ ]:
@dataclass
class SocialGraphDataset:
    name: str
    G: nx.DiGraph                 # u -> v : "u follows v"
    private: dict
    community: dict
    node_list: list
    features: np.ndarray
    meta: dict = field(default_factory=dict)
    def followers(self, t):  return set(self.G.predecessors(t))   # accounts following t (in)
    def following(self, t):  return set(self.G.successors(t))     # accounts t follows (out)
    def neighborhood(self, t): return self.followers(t) | self.following(t)   # N(t) = GROUND TRUTH

## 3. Building blocks: routing, labeling, features

In [ ]:
def orient_edges(G_und, mutual_ratio, rng):
    # undirected base -> directed tracking graph
    D = nx.DiGraph(); D.add_nodes_from(G_und.nodes())
    for u, v in G_und.edges():
        if rng.random() < mutual_ratio:
            D.add_edge(u, v); D.add_edge(v, u)            # mutual
        else:
            if rng.random() < 0.5: D.add_edge(u, v)        # one-way
            else:                  D.add_edge(v, u)
    return D

def assign_private(G, ratio, mode, rng):
    nodes = list(G.nodes()); n_priv = int(round(ratio * len(nodes)))
    if mode == "low_degree":
        deg = np.array([G.degree(n) for n in nodes], dtype=float)
        w = 1.0/(1.0+deg); w /= w.sum()
        chosen = rng.choice(len(nodes), size=n_priv, replace=False, p=w)
    else:
        chosen = rng.choice(len(nodes), size=n_priv, replace=False)
    priv = {n: False for n in nodes}
    for i in chosen: priv[nodes[i]] = True
    return priv

def make_features(node_list, community, dim, homophily, rng):
    # community center + noise; as homophily increases, noise decreases (legitimate signal strengthens)
    comms = sorted(set(community.values()))
    centroids = rng.normal(size=(len(comms), dim))
    centroids /= (np.linalg.norm(centroids, axis=1, keepdims=True) + 1e-9)
    cidx = {c: i for i, c in enumerate(comms)}
    noise = (1.0 - homophily) + 1e-3
    X = np.zeros((len(node_list), dim))
    for i, n in enumerate(node_list):
        X[i] = centroids[cidx[community[n]]] + rng.normal(scale=noise, size=dim)
    return X

## 4. Graph generators (SBM, LFR, real ego-Facebook)

In [ ]:
def gen_sbm(cfg, rng):
    sizes = list(cfg.sbm_sizes); k = len(sizes)
    p = [[cfg.sbm_p_in if i==j else cfg.sbm_p_out for j in range(k)] for i in range(k)]
    G = nx.stochastic_block_model(sizes, p, seed=int(rng.integers(1<<31)))
    return G, {n: G.nodes[n]["block"] for n in G.nodes()}

def gen_lfr(cfg, rng, max_tries=8):
    last = None
    for _ in range(max_tries):
        try:
            G = nx.LFR_benchmark_graph(n=cfg.lfr_n, tau1=cfg.lfr_tau1, tau2=cfg.lfr_tau2,
                                       mu=cfg.lfr_mu, average_degree=cfg.lfr_avg_degree,
                                       min_community=cfg.lfr_min_community, max_iters=1000,
                                       seed=int(rng.integers(1<<31)))
            G.remove_edges_from(nx.selfloop_edges(G))
            raw = {n: frozenset(G.nodes[n]["community"]) for n in G.nodes()}
            uniq = {fs: i for i, fs in enumerate(sorted(set(raw.values()), key=lambda s: min(s)))}
            return G, {n: uniq[raw[n]] for n in G.nodes()}
        except Exception as e:
            last = e
    raise RuntimeError(f"LFR failed in {max_tries} attempts: {last}")

def load_real_edgelist(url, timeout=90):
    raw = urllib.request.urlopen(url, timeout=timeout).read()
    return nx.parse_edgelist(gzip.decompress(raw).decode().strip().split("\n"), nodetype=int)

def bfs_subsample(G, node_cap, rng):
    # Snowball/BFS sampling to reduce large real-world graphs (e.g. Twitter-ego) to a manageable size.
    # Note: introduces a slight bias towards the root node's high-degree neighborhood (standard, explicitly stated).
    if node_cap is None or G.number_of_nodes() <= node_cap:
        return G
    root = int(rng.choice(list(G.nodes())))
    seen = {root}; frontier = [root]
    while frontier and len(seen) < node_cap:
        nxt = []
        for u in frontier:
            for v in G.neighbors(u):
                if v not in seen:
                    seen.add(v); nxt.append(v)
                    if len(seen) >= node_cap: break
            if len(seen) >= node_cap: break
        frontier = nxt
    return G.subgraph(seen).copy()

def load_facebook(cfg, rng):
    G = load_real_edgelist(cfg.snap_url)
    community = {}
    for ci, cset in enumerate(nx.community.greedy_modularity_communities(G)):
        for n in cset: community[n] = ci
    return G, community

def load_real_capped(cfg, rng, url, node_cap):
    # Common loader for Twitter-ego / email-Eu-core (Referee 3, item 1). With facebook_combined
    # same family/format/ethics profile (McAuley & Leskovec, NIPS 2012) or (email-Eu-core) same SNAP source.
    G = load_real_edgelist(url)
    G = bfs_subsample(G, node_cap, rng)
    G = nx.convert_node_labels_to_integers(G)
    community = {}
    for ci, cset in enumerate(nx.community.greedy_modularity_communities(G)):
        for n in cset: community[n] = ci
    return G, community

## 5. Single point of entry: `build_dataset`

In [ ]:
def build_dataset(cfg: DataConfig) -> SocialGraphDataset:
    rng = np.random.default_rng(cfg.seed)
    if   cfg.name.startswith("sbm"): G_und, community = gen_sbm(cfg, rng)
    elif cfg.name.startswith("lfr"): G_und, community = gen_lfr(cfg, rng)
    elif cfg.name.startswith(("facebook","real")): G_und, community = load_facebook(cfg, rng)
    elif cfg.name.startswith("twitter"): G_und, community = load_real_capped(cfg, rng, cfg.twitter_url, cfg.real_node_cap)
    elif cfg.name.startswith("email"):   G_und, community = load_real_capped(cfg, rng, cfg.email_url, None)
    else: raise ValueError(f"Bilinmeyen veri seti: {cfg.name}")

    G_und = nx.convert_node_labels_to_integers(G_und, label_attribute="orig")
    community = {i: community[G_und.nodes[i]["orig"]] for i in G_und.nodes()}

    D = orient_edges(G_und, cfg.mutual_ratio, rng)
    private = assign_private(D, cfg.private_ratio, cfg.private_mode, rng)
    node_list = sorted(D.nodes())
    features = make_features(node_list, community, cfg.feature_dim, cfg.homophily, rng)
    meta = {"config": asdict(cfg), "n_nodes": D.number_of_nodes(), "n_edges": D.number_of_edges()}
    return SocialGraphDataset(cfg.name, D, private, community, node_list, features, meta)

def summarize(ds: SocialGraphDataset) -> dict:
    D = ds.G; und = D.to_undirected(reciprocal=False)
    mutual = sum(1 for u, v in und.edges() if D.has_edge(u, v) and D.has_edge(v, u))
    try:
        from sklearn.metrics import silhouette_score
        labels = np.array([ds.community[n] for n in ds.node_list])
        sil = float(silhouette_score(ds.features, labels)) if len(set(labels)) > 1 else float("nan")
    except Exception:
        sil = float("nan")
    degs = [d for _, d in D.degree()]
    return {"n_nodes": D.number_of_nodes(), "n_directed_edges": D.number_of_edges(),
            "n_relations": und.number_of_edges(),
            "mutual_share": round(mutual/max(und.number_of_edges(),1), 3),
            "n_communities": len(set(ds.community.values())),
            "private_ratio": round(float(np.mean(list(ds.private.values()))), 3),
            "avg_degree": round(float(np.mean(degs)), 2), "max_degree": int(np.max(degs)),
            "feature_silhouette": round(sil, 3)}

## 6. Save to Drive / Read from Drive

Each dataset is written under `DATA_DIR/<name>/`: `edges.csv`, `nodes.csv`, `features.npy`, `graph.pkl`, `meta.json`. These folders are persistent on Google Drive; the next notebook reads them directly with `load_dataset(...)`.

In [ ]:
def save_dataset(ds, root=DATA_DIR):
    outdir = os.path.join(root, ds.name); os.makedirs(outdir, exist_ok=True)
    nx.write_edgelist(ds.G, os.path.join(outdir, "edges.csv"), delimiter=",", data=False)
    pd.DataFrame({"node_id": ds.node_list,
                  "private": [ds.private[n] for n in ds.node_list],
                  "community": [ds.community[n] for n in ds.node_list]}
                 ).to_csv(os.path.join(outdir, "nodes.csv"), index=False)
    np.save(os.path.join(outdir, "features.npy"), ds.features)
    with open(os.path.join(outdir, "graph.pkl"), "wb") as f: pickle.dump(ds.G, f)
    with open(os.path.join(outdir, "meta.json"), "w") as f:
        json.dump({**ds.meta, "summary": summarize(ds)}, f, indent=2)
    return outdir

def load_dataset(name, root=DATA_DIR) -> SocialGraphDataset:
    indir = os.path.join(root, name)
    with open(os.path.join(indir, "graph.pkl"), "rb") as f: G = pickle.load(f)
    ndf = pd.read_csv(os.path.join(indir, "nodes.csv"))
    features = np.load(os.path.join(indir, "features.npy"))
    with open(os.path.join(indir, "meta.json")) as f: meta = json.load(f)
    return SocialGraphDataset(name, G, dict(zip(ndf.node_id, ndf["private"])),
                              dict(zip(ndf.node_id, ndf.community)), list(ndf.node_id), features, meta)

## 7. Generate the datasets and save to Drive

In [ ]:
configs = [DataConfig(name="sbm_default"), DataConfig(name="lfr_default")]
rows = []
for cfg in configs:
    ds = build_dataset(cfg)
    out = save_dataset(ds)
    rows.append({"dataset": cfg.name, **summarize(ds)})
    print("saved ->", out)

pd.DataFrame(rows).set_index("dataset")

## 8. (Optional) Real graph: SNAP ego-Facebook

In [ ]:
try:
    fb = build_dataset(DataConfig(name="facebook_real"))
    print("saved ->", save_dataset(fb))
    print(summarize(fb))
except Exception as e:
    print("Download skipped/failed (can continue with synthetic sets):", e)

## 8b. (NEW) Additional real graphs, Twitter-ego and email-Eu-core (Reviewer 3, item 1)

With a single real dataset (against the "one-time luck" objection), Twitter-ego (same McAuley&Leskovec, NIPS 2012
family, same format/ethics profile as Facebook; reduced to `real_node_cap` (default 5000) nodes via BFS) and
email-Eu-core (the internal email network of a small European research institution; a different RELATIONSHIP TYPE, not social "following"
but communication, so it tests generalization not just with another ego-network but also with another DOMAIN).

In [ ]:
for name in ["twitter_real", "email_real"]:
    try:
        ds = build_dataset(DataConfig(name=name))
        out = save_dataset(ds)
        print("saved ->", out)
        print(summarize(ds))
    except Exception as e:
        print(f"[{name}] download/processing skipped/failed (can continue with other datasets):", e)

## 9. Quick validation: read back from Drive + ground-truth

In [ ]:
ds = load_dataset("sbm_default")     # the next notebook does exactly this
targets = [n for n in ds.node_list if ds.private[n] and ds.G.degree(n) >= 4]
t = targets[0]
print("Read from Drive:", ds.name)
print(f"Private target {t}: followers={len(ds.followers(t))}, following={len(ds.following(t))}, "
      f"N(t)={len(ds.neighborhood(t))} | total candidate private targets={len(targets)}")

## For the next notebook

**02 Recommender Training** will start with the following line (same Drive path, same schema):

```python
from google.colab import drive; drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/cold_start_leakage/data'
ds = load_dataset("sbm_default", DATA_DIR)   # SocialGraphDataset and load_dataset definitions are redeclared here
```

Standard (not designed to leak) LightGCN / GraphSAGE / GraFRank-style models will be built on it, the cold→warm observer axis will be the independent variable. Models will also be saved to Drive (`models/`).